# Visualize

## All

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np

# 设置绘图样式
# sns.set(style="whitegrid", font_scale=1.1)
plt.rcParams.update({
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

# 创建保存目录
os.makedirs("plots", exist_ok=True)

# 读取数据
df = pd.read_csv('res_poisson.csv')

# 数值列转换为 float，防止出现负误差或字符串问题
numeric_cols = ['avg', '25%', '50%', '75%', '1']
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

# 添加 reflection_type 列
df['reflection_type'] = df['K1'].astype(str) + '_' + df['K2'].astype(str)

# 要比较的变量
variables_to_compare = ['prompt_type', 'background_info', 'external_opt']


# 文件名构造函数
def fixed_vars_to_filename(fixed_vars: dict) -> str:
    return "_".join([f"{k}-{v}" for k, v in fixed_vars.items()])


# 绘图函数
def plot_avg_and_1(df, vary_var, fixed_vars):
    sub_df = df.copy()
    for k, v in fixed_vars.items():
        sub_df = sub_df[sub_df[k] == v]
    if sub_df.empty:
        print(f"跳过: {fixed_vars}，因为筛选后无数据")
        return

    unique_vals = sorted(sub_df[vary_var].unique())
    # 增加子图高度，调整宽度
    fig, axes = plt.subplots(1, len(unique_vals),
                             figsize=(6 * len(unique_vals), 8),  # 增加高度到8
                             sharey=True)

    if len(unique_vals) == 1:
        axes = [axes]

    # 计算所有子图的统一x轴范围
    global_min = min(sub_df['25%'].min(), sub_df['1'].min())
    global_max = max(sub_df['75%'].max(), sub_df['1'].max())*0.9
    # 添加一些边距
    x_min = max(0, global_min - 0.1 * (global_max - global_min))
    x_max = global_max + 0.1 * (global_max - global_min)

    for i, val in enumerate(unique_vals):
        ax = axes[i]
        plot_df = sub_df[sub_df[vary_var] == val].copy()

        # # 按照 '1' 排序
        # plot_df = plot_df.sort_values('1', ascending=False)
        # plot_df["reflection_type"] = pd.Categorical(plot_df["reflection_type"],
        #                                             categories=plot_df["reflection_type"],
        #                                             ordered=True)

        y = plot_df["reflection_type"]

        # 绘制 avg
        ax.barh(y=y,
                width=plot_df['avg'],
                color='#ADD8E6',  # 淡蓝色
                label='avg',
                height=0.35,
                align='center',
                left=None)

        # 添加误差线
        for j, row in plot_df.iterrows():
            avg = float(row['avg'])
            p25 = float(row['25%'])
            p75 = float(row['75%'])
            err_low = max(avg - p25, 0)
            err_high = max(p75 - avg, 0)
            ax.errorbar(
                x=avg,
                y=row['reflection_type'],
                xerr=[[err_low], [err_high]],
                fmt='none',
                ecolor='gray',
                capsize=3,
                linewidth=1.2,
                alpha=0.9
            )

        # 绘制 1（错开位置）
        ax.barh(y=plot_df["reflection_type"],
                width=plot_df['1'],
                color='#90EE90',  # 淡绿色
                label='1',
                height=0.35,
                align='center',
                left=None,
                edgecolor='black',
                alpha=0.7,
                hatch='///')  # 可选样式：斜纹区分

        # 设置统一的x轴范围
        ax.set_xlim(x_min, x_max)

        ax.set_title(f"{vary_var} = {val}")
        if i == 0:
            ax.set_ylabel("Reflection Type")
        else:
            ax.set_ylabel("")
            ax.get_yaxis().set_visible(False)

        ax.set_xlabel("Metric Value")

        # 调整图例位置
        if i == len(unique_vals) - 1:
            ax.legend(loc='lower right', bbox_to_anchor=(1.1, 0))
        else:
            legend = ax.get_legend()
            if legend is not None:
                legend.remove()

    fig.suptitle(f"{vary_var} effect (fixed: {fixed_vars})", fontsize=16)
    fig.tight_layout(rect=[0, 0, 1, 0.95])  # 调整布局参数
    filename = f"plots/{vary_var}_fixed_{fixed_vars_to_filename(fixed_vars)}.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ 图像已保存：{filename}")


# 遍历组合绘图
for var in variables_to_compare:
    fixed_vars_list = [v for v in variables_to_compare if v != var]
    unique_settings = df[fixed_vars_list].drop_duplicates()

    for _, setting in unique_settings.iterrows():
        fixed_vars = setting.to_dict()
        plot_avg_and_1(df, var, fixed_vars)

## K1K2

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import seaborn as sns


# 设置图形风格
sns.set(style="whitegrid", font_scale=1.1)
plt.rcParams.update({
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

# 读取数据
df = pd.read_csv("plots/K1K2/K1K2.csv")  # 替换为你的数据路径

# 数值转换，确保没有字符串或异常值
numeric_cols = ['avg', '25%', '75%', '1']
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

# 添加 reflection_type 字段
df["reflection_type"] = df["K1"].astype(str) + "_" + df["K2"].astype(str)

# 遍历每个 group
for group_name, group_df in df.groupby("group"):

    external_opts = sorted(group_df["external_opt"].unique())
    ncols = len(external_opts)
    fig, axes = plt.subplots(1, ncols, figsize=(6 * ncols, 6), sharey=True)

    if ncols == 1:
        axes = [axes]

    # 横轴统一范围
    x_min = min(group_df[["avg", "1"]].min().min(), 0)
    x_max = group_df[["avg", "1"]].max().max()
    x_pad = (x_max - x_min) * 0.1
    xlim = (0, x_max + x_pad)

    for i, ext_opt in enumerate(external_opts):
        ax = axes[i]
        sub_df = group_df[group_df["external_opt"] == ext_opt].copy()
        y = sub_df["reflection_type"]

        # 绘制 avg bar
        ax.barh(
            y=y,
            width=sub_df["avg"],
            color="#ADD8E6",  # 浅蓝
            height=0.35,
            label="avg"
        )

        # 绘制误差线
        for j, row in sub_df.iterrows():
            avg = row["avg"]
            p25 = row["25%"]
            p75 = row["75%"]
            err_low = max(avg - p25, 0)
            err_high = max(p75 - avg, 0)
            ax.errorbar(
                x=avg,
                y=row["reflection_type"],
                xerr=[[err_low], [err_high]],
                fmt='none',
                ecolor='gray',
                capsize=3,
                linewidth=1.2
            )

        # 绘制 1 bar（浅绿）
        ax.barh(
            y=y,
            width=sub_df["1"],
            color="#90EE90",  # 浅绿
            height=0.35,
            label="1",
            alpha=0.8
        )

        ax.set_title(f"external_opt = {ext_opt}")
        ax.set_xlim(xlim)
        if i == 0:
            pass
            # ax.set_ylabel("Reflection Type")
        else:
            ax.set_ylabel("")
            ax.get_yaxis().set_visible(False)

        if i == ncols - 1:
            ax.legend(loc="lower right", bbox_to_anchor=(1.0, 0))

        # ax.set_xlabel("Value")

    fig.suptitle(f"Group: {group_name}", fontsize=16)
    fig.tight_layout(rect=[0, 0, 0.95, 0.95])
    filename = f"plots/K1K2/K1K2_{group_name}.png"
    plt.savefig(filename, dpi=300)
    plt.close()
    print(f"✅ {filename}")


✅ plots/K1K2/K1K2_10a.png
✅ plots/K1K2/K1K2_10b.png
✅ plots/K1K2/K1K2_10c.png
✅ plots/K1K2/K1K2_10d.png
✅ plots/K1K2/K1K2_11a.png
✅ plots/K1K2/K1K2_11b.png
✅ plots/K1K2/K1K2_12a.png
✅ plots/K1K2/K1K2_12b.png
✅ plots/K1K2/K1K2_15a.png
✅ plots/K1K2/K1K2_16a.png
✅ plots/K1K2/K1K2_16a10.png
✅ plots/K1K2/K1K2_16a30.png
✅ plots/K1K2/K1K2_16a50.png
✅ plots/K1K2/K1K2_16b.png
✅ plots/K1K2/K1K2_16b10.png
✅ plots/K1K2/K1K2_16b30.png
✅ plots/K1K2/K1K2_16b50.png
✅ plots/K1K2/K1K2_16c.png
✅ plots/K1K2/K1K2_17a.png
✅ plots/K1K2/K1K2_17b.png


In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import seaborn as sns

# 设置图形风格
sns.set(style="whitegrid", font_scale=1.1)
plt.rcParams.update({
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

# 读取数据
df = pd.read_csv("plots/K1K2/K1K2.csv")  # 替换为你的数据路径

# 数值转换，确保没有字符串或异常值
numeric_cols = ['avg', '25%', '75%', '1']
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

# 添加 reflection_type 字段
df["reflection_type"] = df["K1"].astype(str) + "_" + df["K2"].astype(str)

# ✅ 用户指定要分析的 group 列表
# target_groups = ['10a','10b','10c','10d']  # 替换为你需要的 group 名称列表
# target_groups = ['11a','11b']
# target_groups = ['12a','12b']
# target_groups = ['16a','16b']
# target_groups = ['17a','17b']

# 筛选目标 group 的数据
df_filtered = df[df["group"].isin(target_groups)].copy()

# 平均值计算（按 external_opt 和 reflection_type 分组）
grouped_df = df_filtered.groupby(["external_opt", "reflection_type"])[numeric_cols].mean().reset_index()

# 删除 avg 为 NaN 的行（可选）
grouped_df = grouped_df.dropna(subset=["avg"])

# 获取 external_opt 的唯一值，决定子图数量
external_opts = sorted(grouped_df["external_opt"].unique())
ncols = len(external_opts)

# 创建图形
fig, axes = plt.subplots(1, ncols, figsize=(6 * ncols, 6), sharey=True)
if ncols == 1:
    axes = [axes]

# 统一横轴范围
x_min = min(grouped_df[["avg", "1"]].min().min(), 0)
x_max = grouped_df[["avg", "1"]].max().max()
x_pad = (x_max - x_min) * 0.1
xlim = (0, x_max + x_pad)

for i, ext_opt in enumerate(external_opts):
    ax = axes[i]
    sub_df = grouped_df[grouped_df["external_opt"] == ext_opt].copy()

    # 按照 1 值排序 reflection_type
    # sub_df = sub_df.sort_values("1", ascending=False)
    y = sub_df["reflection_type"]

    # 绘制 avg bar
    ax.barh(
        y=y,
        width=sub_df["avg"],
        color="#ADD8E6",  # 浅蓝
        height=0.35,
        label="avg"
    )

    # 绘制误差线
    for j, row in sub_df.iterrows():
        avg = row["avg"]
        p25 = row["25%"]
        p75 = row["75%"]
        err_low = max(avg - p25, 0)
        err_high = max(p75 - avg, 0)
        ax.errorbar(
            x=avg,
            y=row["reflection_type"],
            xerr=[[err_low], [err_high]],
            fmt='none',
            ecolor='gray',
            capsize=3,
            linewidth=1.2
        )

    # 绘制 1 bar（浅绿）
    ax.barh(
        y=y,
        width=sub_df["1"],
        color="#90EE90",  # 浅绿
        height=0.35,
        label="1",
        alpha=0.8
    )

    ax.set_title(f"external_opt = {ext_opt}")
    ax.set_xlim(xlim)

    if i == 0:
        ax.set_ylabel("")
    else:
        ax.set_ylabel("")
        ax.get_yaxis().set_visible(False)

    if i == ncols - 1:
        ax.legend(loc="lower right", bbox_to_anchor=(1.0, 0))

# 设置总标题
title = f"Group Aggregation: {', '.join(target_groups)}"
fig.suptitle(title, fontsize=16)
fig.tight_layout(rect=[0, 0, 0.95, 0.95])

# 保存图像
os.makedirs("plots", exist_ok=True)
filename = f"plots/K1K2/unsorted_K1K2_aggregated_{'_'.join(target_groups)}.png"
plt.savefig(filename, dpi=300)
plt.close()
print(f"✅ 保存图像到: {filename}")


✅ 保存图像到: plots/K1K2/unsorted_K1K2_aggregated_17a_17b.png
